# Atlas-Guided ViT-UNETR: prototype workflow

End-to-end workflow built **after** `AtlasGuidedUNETR_fixed.ipynb` (the supervised prototype). That notebook
feeds the model its own ground-truth label as the anatomical prior, so its Dice numbers measure a model that is
handed the answer. This notebook replaces that with priors built from 10 reference atlases by registration,
which is what the model would have at prediction time.

| # | Stage | Code | Status |
|---|---|---|---|
| 1 | Data split and leakage control | `data/splits.py` | done |
| 2 | Supervised prototype, distance-map fix, baseline check | `data/transforms.py`, `scripts/evaluate_checkpoint.py` | done |
| 3 | Atlas selection (10 temporary atlases) | `scripts/select_atlas_cases.py` | done |
| 4 | Landmarks | `atlas/landmarks.py` | done |
| 5 | Golden transformation and reliable features | `atlas/golden_transform.py`, `atlas/tps.py`, `atlas/reliable_features.py` | done |
| 6 | Atlas → target registration and label warping | `atlas/matching.py`, `ransac.py`, `registration.py`, `label_warping.py` | done; atlas-only Dice 0.45 to 0.48 on 2 val cases |
| 7 | Priors P(v), D(v), w(v) | `data/transforms.py` (`AtlasPriorsd`) | done |
| 8 | Model and Experiment 1 | `models/`, `training/`, `scripts/train.py` | set up, full run pending on a server |
| 9 | Atlas-consistency and CT-physics losses | none | **not implemented** |

**How to use it.** The cells run top to bottom. Anything slow is behind a `RUN_*` flag in the configuration cell.
With every flag `False` the notebook reads existing artifacts under `cache/` and `experiments/` and only does
cheap computation. Full-scale runs (registering ~670 targets, training 50+ epochs) belong on a server; the exact
commands are in `RUN_GUIDE.md`. Decisions, bugs found and results so far are in `research_audit.md` and
`experiments/registration_log.md`.

**Read the numbers with care.** Every result below comes from a small local chunk (2 to 3 cases), not from the
full validation set. They show that each stage works and roughly how well, not final performance.

Tests are deliberately not part of this notebook; they live in `tests/`.

## 0. Setup

Locates the repository root (so the notebook works from any working directory inside it), prints versions and
the GPU, and defines the run flags.

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "models" / "segmentation_model.py").exists() and (p / "atlas").is_dir():
            return p
    raise RuntimeError("Open this notebook from inside the repository (models/ and atlas/ were not found).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import monai
import SimpleITK as sitk
import matplotlib.pyplot as plt
from monai.utils import set_determinism

set_determinism(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo      :", REPO_ROOT)
print("torch     :", torch.__version__, "| monai", monai.__version__, "| SimpleITK", sitk.Version_VersionString())
print("device    :", device, "-", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU only")

### Configuration and run flags

| Flag | What it does when `True` | Rough cost (this laptop) |
|---|---|---|
| `RUN_BASELINE_EVAL` | re-runs the baseline checkpoint on `BASELINE_LIMIT` validation cases with ground-truth priors | a few min |
| `RUN_ATLAS_SELECTION` | rescans every training label and rewrites `atlas/atlas_selection.json` (deterministic, same result) | ~3 min |
| `RUN_LIBRARY_BUILD` | rebuilds the reliable-landmark library (it is also built automatically if missing) | ~1 min |
| `RUN_REGISTRATION` | registers the 10 atlases onto `N_REGISTER` validation targets and caches the warped labels | ~3 to 4 min per target |
| `RUN_SINGLE_PAIR` | one atlas onto one target in-process, showing every registration stage | ~30 s |
| `RUN_TRAIN_SMOKE` | a few in-process training steps on cached cases (demonstration, nothing is saved) | a few min |
| `RUN_TRAIN_SCRIPT` | launches a tracked training run through `scripts/train.py` (config, git state, metrics saved) | hours, use a server |

In [ ]:
from data.splits import DEFAULT_DATA_DIR

DATA_DIR = DEFAULT_DATA_DIR          # ../Dataset060_Merged_Def, override if yours is elsewhere
IMG_SIZE = (96, 96, 96)
NUM_CLASSES = 13                     # background + 12 organs
ORGAN_IDS = list(range(1, NUM_CLASSES))

RUN_BASELINE_EVAL = False
BASELINE_LIMIT = 3
RUN_ATLAS_SELECTION = False
RUN_LIBRARY_BUILD = False
RUN_REGISTRATION = False
N_REGISTER = 2
RUN_SINGLE_PAIR = True
RUN_TRAIN_SMOKE = False
RUN_TRAIN_SCRIPT = False
TRAIN_SCRIPT_ARGS = ["--run-name", "nb_" + time.strftime("%Y%m%d_%H%M%S"), "--epochs", "50", "--amp"]


def run_script(args):
    # Runs a repo script in a subprocess and streams its output. The heavy, cache-writing steps stay
    # scripts so they can be run unchanged on a server; the notebook only orchestrates and inspects.
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(cmd))
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        if "FutureWarning" in line or "warn_deprecated" in line or line.strip().startswith("warnings.warn"):
            continue
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"{args[0]} exited with code {rc}")

## 1. Data and split

The dataset is 680 paired CT volumes and 12-organ labels from three sources (`amos_*`, `img*`, `s*`). The
train/validation split reproduces the supervised prototype exactly (`random.seed(42)`, shuffle, 80/20), so every
number stays comparable with its epoch-101 result.

Two rules keep the atlas work free of leakage:
- the 10 atlas cases come from the **training** portion and are removed from the training targets, so an atlas
  never gets a supervised loss on the same label that also supplies its prior;
- validation is untouched: the same 136 cases as the baseline.

In [ ]:
from data.splits import train_val_split, split_with_atlases_removed, atlas_names, case_name


def source_of(case):
    return re.match(r"[A-Za-z]+", case_name(case["image"])).group(0)


train_all, val_cases = train_val_split(DATA_DIR)
train_targets, _ = split_with_atlases_removed(DATA_DIR)   # also raises if an atlas is in validation
atlases = atlas_names()

print(f"paired cases      : {len(train_all) + len(val_cases)}")
print(f"train (baseline)  : {len(train_all)}   validation: {len(val_cases)}")
print(f"atlases           : {len(atlases)}   -> train targets: {len(train_targets)}")
print("train targets by source:", dict(Counter(source_of(c) for c in train_targets)))
print("validation by source   :", dict(Counter(source_of(c) for c in val_cases)))
print("first validation cases :", [case_name(c["image"]) for c in val_cases[:3]])

## 2. The supervised prototype, its leakage, and the distance-map fix

In the prototype pipeline (`prior_source="gt"`) the case's own label is one-hot encoded and used as the
probability map P(v). A distance transform of that same label gives D(v). Both go into the model during training
and validation, so the reported Dice (0.6505 at epoch 101) describes a model that can read the answer from its
input. At prediction time on a new scan no label exists, which is the problem the atlas pipeline solves.

In [ ]:
from data.transforms import build_transforms, PRIOR_KEYS

print("which tensors feed the model as (P, D):", PRIOR_KEYS)

tf_gt = build_transforms(IMG_SIZE, NUM_CLASSES, train=False, prior_source="gt", legacy_buggy_distance=True)
sample_gt = tf_gt(val_cases[0])
print({k: tuple(v.shape) for k, v in sample_gt.items() if hasattr(v, "shape")})
print("P(v) for the prototype IS the label: sum over classes =", float(sample_gt["label"].sum(0).mean()),
      "| it is binary:", bool(((sample_gt["label"] == 0) | (sample_gt["label"] == 1)).all()))

### Distance-map bug (found and fixed)

`CustomSpatialDistanceTransformd` called `distance_transform_edt(mask)`. SciPy measures the distance from each
**non-zero** voxel to the nearest zero, so the prototype's D(v) was 0 everywhere outside the organ and grew
towards the organ centre, the opposite of D_c(v) = distance to the organ (0 inside, growing outward). The checkpoint
was trained with this, so `legacy_buggy_distance=True` reproduces it, and all new work uses the fixed default.

In [ ]:
from data.transforms import CustomSpatialDistanceTransformd

size, half, c = 9, 2, 4
block = torch.zeros((3, size, size, size))
block[0] = 1.0
block[0, c - half:c + half + 1, c - half:c + half + 1, c - half:c + half + 1] = 0.0
block[1, c - half:c + half + 1, c - half:c + half + 1, c - half:c + half + 1] = 1.0

def profile(legacy):
    t = CustomSpatialDistanceTransformd(keys=["label"], to_closest_foreground=True, include_background=False,
                                        output_postfix="_distance", legacy_buggy_distance=legacy)
    return t({"label": block})["label_distance"][0][c, c].numpy().round(1)   # line through the organ centre

print("position along a line through the organ (organ occupies indices 2..6)")
print("legacy (buggy):", profile(True))
print("fixed         :", profile(False))

### Baseline reproduction (optional)

`scripts/evaluate_checkpoint.py` runs validation only (the prototype notebook could only resume training). It
needs `--legacy-buggy-distance` for `best_model.pth`. A 3-case run gave Dice 0.6757, consistent with the epoch-101
0.6505; the full 136-case run is meant for a server (see `baseline_reproduction.md`).

In [ ]:
if RUN_BASELINE_EVAL:
    run_script(["scripts/evaluate_checkpoint.py", "--checkpoint", "best_model.pth", "--data-dir", DATA_DIR,
                "--legacy-buggy-distance", "--limit", BASELINE_LIMIT])
else:
    print("skipped (RUN_BASELINE_EVAL = False). Verified earlier: 3 cases -> Dice 0.6757 with ground-truth priors.")

## 3. Atlas selection

The atlases are **temporary**: 10 labelled cases standing in for the doctor-segmented atlases that are still to
come. Replacing them means replacing `atlas/atlas_selection.json` and rebuilding the caches; no code changes.

Selection rules (`scripts/select_atlas_cases.py`): drawn from the training split only; all 12 organs present (in
this merged dataset every training case qualifies); round-robin across the three sources so no single scanner or annotation
protocol dominates; deterministic, no random draw.

In [ ]:
if RUN_ATLAS_SELECTION:
    run_script(["scripts/select_atlas_cases.py", "--data-dir", DATA_DIR])

with open("atlas/atlas_selection.json", encoding="utf-8") as f:
    atlas_cases = json.load(f)["cases"]

val_names = {case_name(c["image"]) for c in val_cases}
train_target_names = {case_name(c["image"]) for c in train_targets}
names = [case_name(c["image"]) for c in atlas_cases]
assert not (set(names) & val_names), "an atlas is in the validation split"
assert not (set(names) & train_target_names), "an atlas is still a training target"

rows = []
for i, c in enumerate(atlas_cases):
    reader = sitk.ImageFileReader()
    reader.SetFileName(c["image"])
    reader.ReadImageInformation()
    rows.append({"atlas": i, "case": case_name(c["image"]), "source": source_of(c),
                 "size (x,y,z)": reader.GetSize(), "spacing mm": tuple(round(s, 2) for s in reader.GetSpacing())})
pd.DataFrame(rows)

The atlases differ a lot in resolution (sub-millimetre in-plane with 5 mm slices, versus 1.5 mm isotropic), which
is why later thresholds are in physical units (mm, mm³) rather than voxels.

## 4. Landmarks

Registration here is landmark based. For each organ of each atlas:
- large organs: points on the surface, chosen by farthest-point sampling so they spread evenly;
- small organs (under 8 mL): a few **interior** points instead. Surfaces are where annotators disagree most, and a
  single centroid gave the reliability score nothing to average over: in an early version organ 11 scored 0 %
  reliable and organ 12 100 % in the same atlas pairs, an all-or-nothing outcome from one point each.

All coordinates are in SimpleITK physical (LPS, mm) space, so atlases with different grids and orientations are
directly comparable. The organ names below are inferred from volume and position (AMOS-style order) and have not
been confirmed against the dataset documentation.

In [ ]:
from atlas.reliable_features import extract_atlas_landmarks_lps

ORGAN_NAMES = {1: "spleen?", 2: "R kidney?", 3: "L kidney?", 4: "gallbladder?", 5: "esophagus?", 6: "liver?",
               7: "stomach?", 8: "aorta?", 9: "IVC?", 10: "pancreas?", 11: "R adrenal?", 12: "L adrenal?"}

t0 = time.time()
lm0 = extract_atlas_landmarks_lps(atlas_cases[0]["label"], ORGAN_IDS)
print(f"landmarks for atlas 0 ({names[0]}) in {time.time() - t0:.1f}s")
print(f"{'organ':>5} {'name (inferred)':<14} {'mode':<9} {'points':>6} {'volume mL':>10}")
for oid, info in sorted(lm0.items()):
    print(f"{oid:>5} {ORGAN_NAMES[oid]:<14} {info['mode']:<9} {len(info['points']):>6} {info['volume_mm3'] / 1000:>10.1f}")

fig = plt.figure(figsize=(13, 5))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2)
cmap = plt.get_cmap("tab20")
for k, (oid, info) in enumerate(sorted(lm0.items())):
    p = info["points"]
    size = 8 if info["mode"] == "surface" else 45
    ax3.scatter(p[:, 0], p[:, 1], p[:, 2], s=size, color=cmap(k))
    ax2.scatter(p[:, 0], p[:, 2], s=size, color=cmap(k), label=str(oid))
ax3.set_xlabel("x (mm)"); ax3.set_ylabel("y (mm)"); ax3.set_zlabel("z (mm)")
ax2.set_xlabel("x (mm)"); ax2.set_ylabel("z (mm)"); ax2.set_aspect("equal")
ax2.set_title("coronal projection"); ax2.legend(ncol=2, fontsize=7, title="organ")
plt.tight_layout(); plt.show()

## 5. Golden transformation and reliable features

Between two fully labelled atlases, correspondence is easy: organ id is an unambiguous key. The golden
transformation (`atlas/golden_transform.py`) does a rigid Kabsch pre-alignment on organ centroids, matches points
per organ by nearest neighbour, and fits a regularised 3D thin-plate spline (`atlas/tps.py`, written from scratch
because SciPy and SimpleITK have none; includes a Jacobian-determinant folding check).

A landmark's residual r under that spline gives a reliability s = max(T − r, 0) / T with T = 15 mm. Averaged over
all 9 other atlases, this is the landmark's score; only landmarks that agree with most other atlases are kept for
matching in unlabelled targets.

The spline's smoothing term is `K − λI` (for the kernel φ(r) = r the matrix K is conditionally negative definite).
An earlier version used `K + λI`, which made every fit erratic; the cell below showed it as `folding: True`
(details in `experiments/registration_log.md`).

In [ ]:
from atlas.golden_transform import build_golden_transformation

lm3 = extract_atlas_landmarks_lps(atlas_cases[3]["label"], ORGAN_IDS)
golden = build_golden_transformation(lm0, lm3, tolerance_mm=15.0, tps_regularization=5.0)   # maps atlas 3 -> atlas 0
r = golden["residuals"]
print(f"atlas {names[3]} -> {names[0]}: {len(r)} correspondences, residual mean {r.mean():.1f} mm, "
      f"median {np.median(r):.1f} mm, reliable (s>0): {golden['reliable_mask'].mean():.0%}")

pts = golden["source_points"][::7]
detj = golden["transform"].jacobian_determinant(pts)
print("Jacobian determinant on a sample of control points:", f"[{detj.min():.2f}, {detj.max():.2f}]",
      "| folding:", bool((detj <= 0).any()))

rows = []
for oid in ORGAN_IDS:
    m = golden["organ_ids"] == oid
    rows.append({"organ": oid, "points": int(m.sum()), "mean residual mm": round(float(r[m].mean()), 1),
                 "reliable": f"{golden['reliable_mask'][m].mean():.0%}"})
pd.DataFrame(rows).set_index("organ").T

In [ ]:
from atlas.reliable_features import load_atlas_library

LIB_PATH = REPO_ROOT / "cache" / "atlas_library" / "atlas_library.npz"
if RUN_LIBRARY_BUILD or not LIB_PATH.exists():
    run_script(["scripts/build_atlas_library.py"])
library = load_atlas_library(str(LIB_PATH))

all_scores = np.concatenate([e["all_scores"] for e in library])
print(f"{len(all_scores)} landmarks over {len(library)} atlases; median score {np.median(all_scores):.2f}, "
      f"p90 {np.percentile(all_scores, 90):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(all_scores, bins=40, color="#4c78a8")
ax[0].axvline(0.3, color="crimson", ls="--", label="keep threshold 0.3")
ax[0].set_xlabel("reliability score"); ax[0].set_ylabel("landmarks"); ax[0].legend()
kept = {oid: np.mean([(e["organ_ids"] == oid).sum() for e in library]) for oid in ORGAN_IDS}
ax[1].bar([str(o) for o in kept], list(kept.values()), color="#59a14f")
ax[1].set_xlabel("organ"); ax[1].set_ylabel("kept landmarks per atlas")
plt.tight_layout(); plt.show()

print("selected / total per atlas:", [f"{len(e['points'])}/{len(e['all_points'])}" for e in library])

With the corrected TPS the score median is 0.43 and the threshold of 0.3 keeps about three quarters of the landmarks
(~300 per atlas), so it is now barely selective. It was picked when the median was 0.32, and is an open tuning item
(to be tuned on training targets, not validation). Organs 11 and 12 keep only ~4 to 5 landmarks each, so the small
organs are the thinnest part of the registration.

## 6. Atlas → target registration and label warping

For an unlabelled target CT, each atlas is registered by (`atlas/registration.py`):

1. **Coarse alignment** on body masks: a z-offset search (scans cover different stretches of the body), then a
   similarity fit, then an affine accepted only if every axis scale stays in 0.8 to 1.25. An earlier unconstrained
   affine collapsed one atlas to 0.19× height when the fields of view differed.
2. **Patch matching**: each reliable atlas landmark is located in the target by normalised cross-correlation of a
   9³-voxel soft-tissue patch in a local window (a batched GPU convolution, `atlas/matching.py`), in two passes with a
   tighter second window.
3. **RANSAC affine** over those matches, rejecting mirror images and implausible scales (`atlas/ransac.py`).
4. **Smoothing TPS** on the inliers, fitted target → atlas so labels can be pulled back without inverting a spline,
   with a folding check and an explicit, recorded fallback (`tps`, `ransac_affine_only`, `coarse_affine_only`). The
   smoothing weight λ starts at 5 and is multiplied by 4 while the field folds; so far every registration has stayed at 5.
5. **Label warping**: the atlas label is resampled into the target's 2 mm grid by nearest neighbour; the 10 warped
   labels are cached as one 4D file per target (`atlas/label_warping.py`).

The target's own label is never used. With `--eval-gt` it is read afterwards, only to report how good the
atlas-only vote is.

In [ ]:
if RUN_REGISTRATION:
    run_script(["scripts/register_targets.py", "--split", "val", "--limit", N_REGISTER, "--eval-gt", "--qa-png"])

reg_dir = REPO_ROOT / "cache" / "registrations" / "tps"
records = sorted(reg_dir.glob("*.json")) if reg_dir.exists() else []
rows = []
for p in records:
    rec = json.loads(p.read_text(encoding="utf-8"))
    ev = rec.get("evaluation_only")
    row = {"case": p.stem, "split": "val" if p.stem in val_names else "train", **rec["status_counts"],
           "time s": rec["elapsed_s"]}
    if ev:
        row["atlas-only Dice"] = round(ev["majority_vote_mean_dice"], 3)
        row.update({f"o{k}": round(v, 2) for k, v in ev["majority_vote_dice"].items()})
    rows.append(row)
registration_table = pd.DataFrame(rows)
print(f"{len(records)} cached registrations in {reg_dir}")
registration_table

In [ ]:
qa_dir = REPO_ROOT / "cache" / "qa" / "tps"
pngs = sorted(qa_dir.glob("*.png"))[:2] if qa_dir.exists() else []
for p in pngs:
    plt.figure(figsize=(16, 6.5))
    plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(p.stem)
    plt.tight_layout(); plt.show()
if not pngs:
    print("no QA images yet: set RUN_REGISTRATION = True (it writes them)")

### One atlas, one target, every stage

Runs a single registration in-process so the intermediate quantities are visible: what the coarse stage chose,
how many landmark matches survive thresholding and RANSAC in each pass, and whether the TPS needed extra smoothing
to avoid folding.

In [ ]:
from atlas.geometry import read_ct, resample_isotropic
from atlas.registration import RegistrationConfig, prepare_target, register_atlas_to_target

reg = None
if RUN_SINGLE_PAIR:
    cfg = RegistrationConfig()
    target_case = val_cases[0]
    t0 = time.time()
    prep = prepare_target(read_ct(target_case["image"]), cfg)
    atlas_ct = resample_isotropic(read_ct(library[0]["image"]), cfg.match_spacing_mm, sitk.sitkLinear, -1000.0)
    reg = register_atlas_to_target(atlas_ct, library[0]["points"], prep, cfg)
    q = reg.qa
    print(f"target {case_name(target_case['image'])}  <-  atlas {names[0]}   ({time.time() - t0:.0f}s)")
    c = q["coarse"]
    print(f"coarse    : stage={c['stage']}, z offset {c['z_offset_mm']:.0f} mm (overlap {c['z_overlap']:.0%}), "
          f"axis scales {[round(s, 2) for s in c['affine_scales']]}")
    for key in ("pass1", "pass2"):
        if key in q:
            s = q[key]
            print(f"{key}     : window +-{s['search_radius']} vox, {s['num_candidates']} candidates, "
                  f"{s['ransac'].get('num_inliers', 0)} RANSAC inliers, median NCC {s['median_ncc_candidates']:.2f}")
    print(f"used pass {q['pass_used']} | status: {reg.status} | TPS lambda {q.get('tps_regularization_used')} "
          f"| fold fraction {q.get('fold_fraction')}")

    d = reg.debug
    plt.figure(figsize=(6, 3.2))
    plt.hist(d["ncc"], bins=40, alpha=0.6, label="all landmarks")
    plt.hist(d["ncc"][d["inliers"]], bins=40, alpha=0.8, label="RANSAC inliers")
    plt.axvline(cfg.min_ncc, color="k", ls="--", label="NCC threshold")
    plt.xlabel("peak NCC of best patch match"); plt.ylabel("landmarks"); plt.legend()
    plt.tight_layout(); plt.show()
else:
    print("skipped (RUN_SINGLE_PAIR = False)")

### What the registration currently achieves

Atlas-only majority vote on the first two validation targets (evaluation only, target labels never used to build
the priors); details and open tuning items in `experiments/registration_log.md`.

| Variant | amos_0278 | s0703 |
|---|---|---|
| first pipeline (unconstrained coarse affine) * | 0.334 | 0.328 |
| constrained coarse stage, second matching pass * | 0.321 | 0.321 |
| + B-spline refinement on whole-body mutual information * | 0.265 | 0.316 |
| **after fixing the TPS smoothing sign (current)** | **0.445** | **0.481** |
| current + B-spline refinement | 0.443 | 0.458 |

\* produced with the faulty TPS, kept only for history.

What this says:
- The TPS sign error had hidden the TPS's benefit: fixing it raised atlas-only Dice from about 0.32 to 0.45 to 0.48
  with no other change. For a single atlas on amos_0278 the stages now order coarse affine < RANSAC affine < TPS
  (for example 0.36 → 0.44 → 0.54).
- 49 of 50 cached registrations end in a TPS (one falls back to the coarse affine), all at the first smoothing value
  and without folding.
- Dense B-spline refinement does not help: whole-body mutual information improves in every pair while organ overlap
  does not, so it is not used (it stays available as `--refinement bspline`).
- Still weak: organ 4 is 0.00 on both cases and organs 11 and 12 are near 0 on one of them. Two cases say little
  about individual organs.
- Open, all to be tuned on non-atlas *training* targets: golden-transform λ and tolerance, reliability threshold,
  registration TPS λ, RANSAC threshold and patch size.

## 7. Priors: P(v), D(v), w(v)

`AtlasPriorsd` (in `data/transforms.py`) turns the cached warped-atlas stack into the model's inputs **after** all
cropping, resizing and augmentation, so the priors stay aligned with the CT the model actually sees:

- P_c(v): fraction of the 10 atlases that label voxel v as organ c,
- w(v) = 1 − H(v)/log C: confidence from the entropy of that vote,
- D_c(v): distance in mm to the atlas-derived region of organ c, clipped at 50 mm and divided by 50.

The organ region is where P_c is at least half its maximum. That rule keeps D defined for small organs whose atlases
rarely agree (a fixed 0.5 cut would leave them empty). The warped stack is loaded by MONAI onto the same 96³ grid as
the image through its tracked affine (`ResampleToMatchd`), then flipped and rotated together with it.

In [ ]:
from data.splits import attach_atlas_priors

cached_val, missing_val = attach_atlas_priors(val_cases, "tps")
print(f"{len(cached_val)} of {len(val_cases)} validation cases have cached atlas stacks")

tf_atlas = build_transforms(IMG_SIZE, NUM_CLASSES, train=False, prior_source="atlas")
demo_case = demo = None
if cached_val:
    demo_case = cached_val[0]
    demo = tf_atlas(demo_case)
    P, D, W = demo["prob"], demo["dist"], demo["conf"]
    print(f"case {case_name(demo_case['image'])}: image {tuple(demo['image'].shape)}, P {tuple(P.shape)}, "
          f"D {tuple(D.shape)}, w {tuple(W.shape)}")
    print(f"P sums to {float(P.sum(0).mean()):.3f} | D in [{float(D.min()):.2f}, {float(D.max()):.2f}] "
          f"| w in [{float(W.min()):.2f}, {float(W.max()):.2f}], mean {float(W.mean()):.2f}")
    print("keys the model sees:", [k for k in demo if k in ("image", "prob", "dist", "conf")],
          "| label only as loss target:", "label" in demo)
else:
    print("no cached stacks: set RUN_REGISTRATION = True first")

In [ ]:
if demo is not None:
    img = demo["image"][0].numpy()
    lab = demo["label"].argmax(0).numpy()
    voted = demo["prob"].argmax(0).numpy()
    conf = demo["conf"][0].numpy()
    dist6 = demo["dist"][5].numpy()                    # channel 5 = organ 6 (distance to its atlas region)

    organ_voxels = np.argwhere(voted > 0)              # slice through where the PRIOR puts organs, not the label
    cx, cy, cz = organ_voxels.mean(axis=0).round().astype(int)
    views = {"axial": lambda a: a[:, :, cz].T, "coronal": lambda a: a[:, cy, :].T}

    fig, axes = plt.subplots(2, 5, figsize=(17, 7))
    for r, (vname, sl) in enumerate(views.items()):
        panels = [("CT", sl(img), "gray", None), ("supervised target (label)", sl(lab), "nipy_spectral", (0, 12)),
                  ("atlas vote P argmax", sl(voted), "nipy_spectral", (0, 12)),
                  ("confidence w(v)", sl(conf), "magma", (0, 1)), ("D for organ 6", sl(dist6), "viridis", (0, 1))]
        for c, (title, arr, cmapname, lim) in enumerate(panels):
            a = axes[r, c]
            a.imshow(arr, cmap=cmapname, origin="lower", **({"vmin": lim[0], "vmax": lim[1]} if lim else {}))
            a.set_title(f"{vname}: {title}", fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()

    dice = []
    for k in ORGAN_IDS:
        p, g = voted == k, lab == k
        dice.append(2 * (p & g).sum() / max(p.sum() + g.sum(), 1))
    plt.figure(figsize=(7, 3))
    plt.bar([str(k) for k in ORGAN_IDS], dice, color="#e15759")
    plt.ylim(0, 1); plt.xlabel("organ"); plt.ylabel("Dice")
    plt.title(f"atlas-only vote vs label in the 96^3 training space (evaluation only), mean {np.mean(dice):.2f}")
    plt.tight_layout(); plt.show()
    print(f"mean atlas-only Dice in the 96^3 training space: {np.mean(dice):.3f}  (2 mm registration grid, same case: "
          f"{registration_table.set_index('case').loc[case_name(demo_case['image']), 'atlas-only Dice']})")

The mean here is close to, but a little below, the value on the 2 mm registration grid (0.42 versus 0.445 for this
case in the last run). That shows warping, cropping, resizing and augmentation keep the prior aligned with the CT; the
coarser 96³ grid probably costs small organs a little, which was not investigated. The prior is informative for large organs but far from the label
for small ones; how much the network can do with it is what Experiment 1 tests.

## 8. Model and Experiment 1

The architecture is unchanged from the prototype: `AtlasGuidedViTUNETR` is a MONAI UNETR whose 12 transformer
blocks use cross-attention, with image tokens as queries and **anatomical tokens** as keys and values. An MLP
builds one anatomical token per 16³ patch from the pooled [P, D] values. Loss (`DiceCELoss`), optimiser, learning
rate and augmentation are also the prototype's, so a change in Dice can be attributed to the prior alone.

In [ ]:
from models.segmentation_model import AtlasGuidedViTUNETR

model = AtlasGuidedViTUNETR(in_channels=1, out_channels=NUM_CLASSES, img_size=IMG_SIZE, feature_size=16,
                            hidden_size=768, mlp_dim=3072, num_heads=12, proj_type="perceptron",
                            norm_name="instance", res_block=True).to(device)
ckpt = torch.load("best_model.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)
print(f"epoch-101 weights loaded, {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")

### Same weights, same case, two kinds of prior

The epoch-101 network evaluated on one validation case, once with its ground-truth prior (what it was trained on)
and once with the atlas prior (what it would get on a new scan). Note this is a single case.

In [ ]:
from monai.data import DataLoader, Dataset
from training.engine import validate, train_one_epoch

def one_case_dice(sample, keys):
    mean, per_organ = validate(model, DataLoader(Dataset([sample]), batch_size=1), device, keys, amp=False)
    return mean, per_organ

if demo is not None:
    plain_case = {k: demo_case[k] for k in ("image", "label")}
    sample_gt = tf_gt(plain_case)
    d_gt, per_gt = one_case_dice(sample_gt, PRIOR_KEYS["gt"])
    d_atlas, per_atlas = one_case_dice(demo, PRIOR_KEYS["atlas"])
    print(f"case {case_name(demo_case['image'])} | epoch-101 weights")
    print(f"  ground-truth prior (prototype) : mean Dice {d_gt:.3f}")
    print(f"  atlas prior                    : mean Dice {d_atlas:.3f}")
    pd.DataFrame({"GT prior": per_gt, "atlas prior": per_atlas}, index=[f"o{k}" for k in ORGAN_IDS]).round(2).T
else:
    print("needs a cached registration (see section 6)")

In the local smoke test the epoch-101 weights scored **0.12 with atlas priors on two validation cases, versus 0.68
with ground-truth priors on the first three (0.65 on the full set)**. Different case counts, so only the gap is
meaningful. The prototype mostly learned to read the prior, so **0.6505 cannot be compared with any atlas-prior
result**. That 0.12 did not move when the prior itself improved (0.120 before the TPS fix, 0.119 after, while atlas-only
Dice rose from about 0.32 to about 0.46); an untested explanation is that the model sees priors only through 16³-patch
pooling and was tuned to exact binary priors. Experiment 1 fine-tunes from these weights to see how far the network
can adapt; a model trained with no prior at all (stock UNETR) is still needed as the honest reference.
(`Atlas_Comparison_Models.ipynb` defines stock UNETR and UNet runs but only with a 5-epoch setup, so that
reference does not exist yet.)

### A few training steps, in-process (demonstration only)

Mirrors the first steps of `scripts/train.py` on cached training targets, on this 4 GB GPU with fp16 autocast.
Nothing is saved, and it **modifies the `model` object in this notebook**, so re-run the model cell to get the
epoch-101 weights back.

In [ ]:
from monai.losses import DiceCELoss

if RUN_TRAIN_SMOKE:
    train_cached, _ = attach_atlas_priors(train_targets, "tps")
    val_subset = cached_val[:2]
    train_subset = train_cached[:3]
    print(f"{len(train_subset)} training targets and {len(val_subset)} validation cases with cached priors")

    val_loader = DataLoader(Dataset(val_subset, tf_atlas), batch_size=1)
    train_loader = DataLoader(Dataset(train_subset, build_transforms(IMG_SIZE, NUM_CLASSES, train=True,
                                                                     prior_source="atlas")),
                              batch_size=1, shuffle=True)
    criterion = DiceCELoss(to_onehot_y=False, softmax=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    keys = PRIOR_KEYS["atlas"]

    before, _ = validate(model, val_loader, device, keys, amp=use_amp)
    loss = train_one_epoch(model, train_loader, optimizer, criterion, device, keys, scaler=scaler, amp=use_amp,
                           max_steps=3, log_every=1)
    after, per = validate(model, val_loader, device, keys, amp=use_amp)
    print(f"validation Dice with atlas priors: {before:.3f} before  ->  {after:.3f} after 3 steps (train loss {loss:.3f})")
    print("per organ after:", {f"o{k}": None if s is None else round(s, 2) for k, s in zip(ORGAN_IDS, per)})
    torch.cuda.empty_cache()
else:
    print("skipped (RUN_TRAIN_SMOKE = False). Local smoke test earlier: 0.119 -> 0.198 after 3 steps on 3 targets "
          "(2 validation cases), which shows the loop works, not that it converges.")

### Tracked runs

`scripts/train.py` is what real runs use. Each run gets its own folder `experiments/<experiment>/<run_name>/` with
`config.json`, `command.txt`, `git.txt` and diff, `metrics.csv`, `summary.json`, `training.log` and checkpoints, and an
existing run folder is never overwritten. Epoch 0 in `metrics.csv` is the epoch-101 weights scored with atlas priors
before any fine-tuning, the reference point for how much the prototype relied on ground-truth priors. The script
refuses to start if some cases have no registration cache (`--allow-missing` is for smoke tests).

Full Experiment 1 needs every target registered first (about 670 targets at ~2 min each, sharded across processes
with `--num-shards`/`--shard-index`); the commands are in `RUN_GUIDE.md`. A run costs hours, so use a server.

In [ ]:
if RUN_TRAIN_SCRIPT:
    run_script(["scripts/train.py", *TRAIN_SCRIPT_ARGS])

frames = {}
for p in sorted((REPO_ROOT / "experiments").glob("*/*/metrics.csv")):
    frames[f"{p.parent.parent.name}/{p.parent.name}"] = pd.read_csv(p)
print("runs found:", list(frames) or "none")
for name, df in frames.items():
    print("\n" + name)
    print(df[["epoch", "train_loss", "mean_dice"]].round(4).to_string(index=False))

## 9. What is done, found, and still open

**Bugs and problems found along the way** (all in `research_audit.md`)
1. The prototype's P(v) and D(v) came from each case's own ground truth: leakage in training and validation.
2. The distance map was inverted (fixed; `legacy_buggy_distance=True` reproduces the old behaviour).
3. `RandRotated(range_x=(-15, 15))` is in **radians**, so the prototype trained with arbitrary rotations rather than
   ±15°. Kept as the default so Experiment 1 changes only the prior; `--rotate-range-deg 15` is the corrected ablation.
4. The label is resized with PyTorch `nearest`, which sits about 0.5 to 0.9 voxel off the CT (resized with `area`).
   It affects the prototype too; documented, not changed.
5. `requirements.txt` originally resolved to a CPU-only torch; it now pins the CUDA build.
6. A sign error in my TPS smoothing term (`K + λI` instead of `K − λI`) made every fit erratic and had hidden the TPS's
   benefit. Found while validating this notebook; atlas-only Dice went from about 0.32 to 0.45 to 0.48 after the fix.

**Still open**
- **Registration quality**: atlas-only Dice is 0.45 to 0.48 on two validation cases; tune the golden-transform λ,
  reliability threshold, TPS λ, RANSAC threshold and patch size on non-atlas *training* targets (never on
  validation). Small organs stay weak.
- **Does the model even see registration detail?** The priors enter only through 16³-patch pooling. A sensitivity test
  (shift the priors by known amounts and watch Dice) would show whether ~10 mm registration gains can matter at all.
- **Experiment 1 at scale**: register all targets and train on a server; compare against a no-prior reference model
  and against the epoch-101 baseline run with ground-truth priors.
- **Doctor-segmented atlases**: replace the 10 temporary atlases when available (one file plus cache rebuild).
- **Experiments 2 to 4 are not implemented**: confidence-weighted atlas-consistency loss (w(v) is already
  produced), CT edge loss, and an HU loss only if it can be justified, since intensities are rescaled to [0, 1].
- **Not from the brief yet**: the `run.py` orchestrator and the `config/` files.